
# Synthetic Audiences - Aberje MVP

Arquitetura otimizada:

- 1 chamada por agente × notícia
- Structured Output (Pydantic)
- GPT-4o-mini
- ThreadPoolExecutor
- REPEATS = 1
- Notícias em DataFrame (`df_news`)
- Agentes em DataFrame (`df_agents`)


## 1. Schema

In [ ]:

from pydantic import BaseModel, Field

class SurveyResponse(BaseModel):
    VAL_01:int = Field(ge=1, le=7)
    VAL_02:int = Field(ge=1, le=7)
    VAL_03:int = Field(ge=1, le=7)
    VAL_04:int = Field(ge=1, le=7)

    ARO_01:int = Field(ge=1, le=7)
    ARO_02:int = Field(ge=1, le=7)
    ARO_03:int = Field(ge=1, le=7)

    POS_01:int = Field(ge=1, le=7)
    POS_02:int = Field(ge=1, le=7)
    POS_03:int = Field(ge=1, le=7)
    POS_04:int = Field(ge=1, le=7)
    POS_05:int = Field(ge=1, le=7)

    NEG_01:int = Field(ge=1, le=7)
    NEG_02:int = Field(ge=1, le=7)
    NEG_03:int = Field(ge=1, le=7)
    NEG_04:int = Field(ge=1, le=7)
    NEG_05:int = Field(ge=1, le=7)

    CRE_01:int = Field(ge=1, le=7)
    CRE_02:int = Field(ge=1, le=7)
    CRE_03:int = Field(ge=1, le=7)

    ATT_01:int = Field(ge=1, le=7)
    ATT_02:int = Field(ge=1, le=7)
    ATT_03:int = Field(ge=1, le=7)
    ATT_04:int = Field(ge=1, le=7)


## 2. Agent Context

In [ ]:

def build_agent_context(agent_row):

    return f"""
PERSONA

{agent_row["Persona"]}

SOCIODEMOGRÁFICO

Idade: {agent_row["idade"]}
Gênero: {agent_row["gênero"]}
Escolaridade: {agent_row["escolaridade"]}
Renda: {agent_row["renda familiar mensal"]}
UF: {agent_row["uf"]}

VALORES E ATITUDES

Identificação política:
{agent_row["identificação política"]}

Ideologia:
{agent_row["ideologia"]}

Segmento:
{agent_row["segmento_e_bolhas"]}

Tipo de fonte consumida:
{agent_row["tipo_de_fontes_de_informação"]}
"""


## 3. Prompt

In [ ]:

PROMPT_TEMPLATE = """
Você é a pessoa descrita abaixo.

{agent_context}

NOTÍCIA

Título:
{title}

Conteúdo:
{content}

Fonte:
{source_type}

Responda TODOS os itens usando números inteiros de 1 a 7.

Retorne SOMENTE JSON.
"""


## 4. Chain

In [ ]:

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2,
)

structured_llm = llm.with_structured_output(
    SurveyResponse
)


## 5. Evaluate News

In [ ]:

def evaluate_news_for_agent(
    agent_row,
    news_row,
):

    prompt = PROMPT_TEMPLATE.format(
        agent_context=build_agent_context(agent_row),
        title=news_row["titulo"],
        content=news_row["conteudo"],
        source_type=news_row["tipo_de_fontes_de_informação"],
    )

    response = structured_llm.invoke(prompt)

    result = response.model_dump()

    result["id_persona"] = agent_row["id_persona"]
    result["news_id"] = news_row["news_id"]

    return result


## 6. Parallel Runner

In [ ]:

from concurrent.futures import ThreadPoolExecutor

def process_task(task):

    agent_row, news_row = task

    return evaluate_news_for_agent(
        agent_row,
        news_row,
    )

def run_population_news(
    df_agents,
    df_news,
    max_workers=30,
):

    tasks = [
        (agent_row, news_row)
        for _, news_row in df_news.iterrows()
        for _, agent_row in df_agents.iterrows()
    ]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:

        results = list(
            executor.map(
                process_task,
                tasks,
            )
        )

    return results


## 7. Run

In [ ]:

results = run_population_news(
    df_agents=df_agents,
    df_news=df_news,
    max_workers=30,
)

df_responses = pd.DataFrame(results)
df_responses.head()


## 8. Construct Scores

In [ ]:

def compute_construct_scores(df):

    df = df.copy()

    df["Valencia"] = df[["VAL_01","VAL_02","VAL_03","VAL_04"]].mean(axis=1)

    df["Arousal"] = df[["ARO_01","ARO_02","ARO_03"]].mean(axis=1)

    df["Emocao_Positiva"] = df[
        ["POS_01","POS_02","POS_03","POS_04","POS_05"]
    ].mean(axis=1)

    df["Emocao_Negativa"] = df[
        ["NEG_01","NEG_02","NEG_03","NEG_04","NEG_05"]
    ].mean(axis=1)

    df["Credibilidade"] = df[
        ["CRE_01","CRE_02","CRE_03"]
    ].mean(axis=1)

    df["Atratividade"] = df[
        ["ATT_01","ATT_02","ATT_03","ATT_04"]
    ].mean(axis=1)

    return df


## 9. Segment Analysis

In [ ]:

df_final = compute_construct_scores(df_responses)

segment_view = (
    df_final.groupby(
        ["news_id", "segmento_e_bolhas"]
    )[
        [
            "Valencia",
            "Arousal",
            "Credibilidade",
            "Atratividade",
        ]
    ]
    .mean()
    .reset_index()
)

segment_view.head()
